## Building the dataset with historical NBA player game logs.

### Install packages/libraries

In [22]:
# !pip install nba_api pandas numpy scikit-learn matplotlib seaborn jupyter requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
from pathlib import Path

from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
from sklearn.metrics import brier_score_loss, accuracy_score

from sklearn.calibration import calibration_curve

#### Retrieve historical game logs for a single NBA player

In [2]:
# Find Luka Doncic's player ID
player = players.find_players_by_full_name("Luka Doncic")[0]
player_id = player["id"]

# Retrieve historical games
game_log = playergamelog.PlayerGameLog(
    player_id=player_id,
    season="2025-26",
    timeout=60
)

df = game_log.get_data_frames()[0]

# Inspect the dataset
print(df.shape)
display(df.head())
print(df.columns.tolist())

(64, 27)


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,1629029,0022501116,"Apr 02, 2026",LAL @ OKC,L,26,3,10,0.300,...,4,4,7,1,0,6,0,12,-25,1
1,22025,1629029,0022501103,"Mar 31, 2026",LAL vs. CLE,W,34,13,26,0.500,...,5,5,12,2,0,0,4,42,21,1
2,22025,1629029,0022501075,"Mar 27, 2026",LAL vs. BKN,W,39,15,25,0.600,...,6,8,3,3,1,5,5,41,19,1
3,22025,1629029,0022501052,"Mar 25, 2026",LAL @ IND,W,38,15,30,0.500,...,5,6,7,1,0,5,5,43,6,1
4,22025,1629029,0022501038,"Mar 23, 2026",LAL @ DET,L,39,11,29,0.379,...,6,7,6,3,0,3,1,32,7,1


['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']


Calculate rolling statistics and construct our binary target.

In [3]:
# Each game's features should only use information from games played before it.
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])

df = df.sort_values("GAME_DATE").reset_index(drop=True)

In [4]:
# Last 5 games point average
df["PTS_L5"] = (
    df["PTS"]
    .shift(1)
    .rolling(window=5)
    .mean()
)

# Last 10 games point average
df["PTS_L10"] = (
    df["PTS"]
    .shift(1)
    .rolling(window=10)
    .mean()
)

# Last 5 games point standard deviation
df["PTS_STD_L5"] = (
    df["PTS"]
    .shift(1)
    .rolling(window=5)
    .std()
)

# Last 10 games point standard deviation
df["PTS_STD_L10"] = (
    df["PTS"]
    .shift(1)
    .rolling(window=10)
    .std()
)

# Rate of games where points are at least 25
df["PTS_25_RATE_L10"] = (
    (df["PTS"] >= 25)
    .shift(1)
    .rolling(window=10)
    .mean()
)

# Last 10 games minute average
df["MIN_L10"] = (
    df["MIN"]
    .shift(1)
    .rolling(window=10)
    .mean()
)

In [5]:
# Binary target
df["TARGET_25"] = (df["PTS"] >= 25).astype(int)

In [6]:
# Sanity check
cols = [
    "GAME_DATE",
    "PTS",
    "PTS_L5",
    "PTS_L10",
    "PTS_STD_L5",
    "PTS_STD_L10",
    "PTS_25_RATE_L10",
    "MIN_L10"
]

display(df[cols].tail(10))

,GAME_DATE,PTS,PTS_L5,PTS_L10,PTS_STD_L5,PTS_STD_L10,PTS_25_RATE_L10,MIN_L10
54,2026-03-14,30,37.6,33.2,9.787747,9.330952,0.9,35.5
55,2026-03-16,36,38.2,34.0,9.038805,8.576454,1.0,36.0
56,2026-03-18,40,36.6,33.5,8.443933,8.263037,1.0,36.1
57,2026-03-19,60,37.6,34.9,8.502941,8.033956,1.0,37.0
58,2026-03-21,33,43.4,38.1,12.033287,10.857153,1.0,37.9
59,2026-03-23,32,39.8,38.7,11.882761,10.328493,1.0,37.9
60,2026-03-25,43,40.2,39.2,11.497826,9.807027,1.0,37.8
61,2026-03-27,41,41.6,39.1,11.282730,9.757618,1.0,38.4
62,2026-03-31,42,41.8,39.7,11.256109,9.661493,1.0,38.5
63,2026-04-02,12,38.2,40.8,5.263079,9.174845,1.0,38.4


In [7]:
print(df["PTS_25_RATE_L10"].describe())

count    54.000000
mean      0.864815
std       0.108432
min       0.700000
25%       0.800000
50%       0.900000
75%       0.900000
max       1.000000
Name: PTS_25_RATE_L10, dtype: float64


Download data

In [8]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "single_player.csv", index=False)

#### Build multi-player dataset

In [14]:
player_names = [
    "Luka Doncic",
    "Shai Gilgeous-Alexander",
    "Nikola Jokic",
    "Jayson Tatum",
    "Anthony Edwards",
    "Devin Booker",
    "Donovan Mitchell",
    "Stephen Curry",
    "Giannis Antetokounmpo",
    "Jalen Brunson"
]

all_games = []

for name in player_names:
    matches = players.find_players_by_full_name(name)

    if not matches:
        print(f"Player not found: {name}")
        continue

    player = matches[0]

    try:
        game_log = playergamelog.PlayerGameLog(
            player_id=player["id"],
            season="2025-26",
            timeout=60
        )

        player_df = game_log.get_data_frames()[0]

        player_df["PLAYER_NAME"] = name
        player_df["PLAYER_ID"] = player["id"]

        all_games.append(player_df)

        print(f"{name}: {len(player_df)} games")

    except Exception as e:
        print(f"Error fetching {name}: {e}")

    time.sleep(1)

df = pd.concat(all_games, ignore_index=True)

df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])

df = df.sort_values(
    ["PLAYER_ID", "GAME_DATE"]
).reset_index(drop=True)

print(f"\nTotal games: {len(df)}")
print(f"Total players: {df['PLAYER_ID'].nunique()}")

Luka Doncic: 64 games
Shai Gilgeous-Alexander: 68 games
Nikola Jokic: 65 games
Jayson Tatum: 16 games
Anthony Edwards: 61 games
Devin Booker: 64 games
Donovan Mitchell: 70 games
Stephen Curry: 43 games
Giannis Antetokounmpo: 36 games
Jalen Brunson: 74 games

Total games: 561
Total players: 10


In [15]:
print(df.groupby("PLAYER_NAME").size())

display(
    df[
        [
            "PLAYER_NAME",
            "GAME_DATE",
            "MATCHUP",
            "PTS",
            "MIN"
        ]
    ].head(15)
)

PLAYER_NAME
Anthony Edwards            61
Devin Booker               64
Donovan Mitchell           70
Giannis Antetokounmpo      36
Jalen Brunson              74
Jayson Tatum               16
Luka Doncic                64
Nikola Jokic               65
Shai Gilgeous-Alexander    68
Stephen Curry              43
dtype: int64


,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,MIN
0,Stephen Curry,2025-10-21,GSW @ LAL,23,32
1,Stephen Curry,2025-10-23,GSW vs. DEN,42,37
2,Stephen Curry,2025-10-24,GSW @ POR,35,27
3,Stephen Curry,2025-10-27,GSW vs. MEM,16,30
4,Stephen Curry,2025-10-28,GSW vs. LAC,19,26
5,Stephen Curry,2025-10-30,GSW @ MIL,27,35
6,Stephen Curry,2025-11-01,GSW @ IND,24,29
7,Stephen Curry,2025-11-04,GSW vs. PHX,28,34
8,Stephen Curry,2025-11-11,GSW @ OKC,11,20
9,Stephen Curry,2025-11-12,GSW @ SAS,46,34


Calculating rolling statistics.

In [16]:
grouped = df.groupby("PLAYER_ID")["PTS"]

df["PTS_L5"] = grouped.transform(
    lambda x: x.shift(1).rolling(5).mean()
)

df["PTS_L10"] = grouped.transform(
    lambda x: x.shift(1).rolling(10).mean()
)

df["PTS_STD_L5"] = grouped.transform(
    lambda x: x.shift(1).rolling(5).std()
)

df["PTS_STD_10"] = grouped.transform(
    lambda x: x.shift(1).rolling(10).std()
)

df["PTS_25_RATE_L10"] = grouped.transform(
    lambda x: (x >= 25).shift(1).rolling(10).mean()
)

df["MIN_L10"] = df.groupby("PLAYER_ID")["MIN"].transform(
    lambda x: pd.to_numeric(x, errors="coerce")
    .shift(1).rolling(10).mean()
)

df["PTS_TREND"] = df["PTS_L5"] - df["PTS_L10"]

df["TARGET_25"] = (df["PTS"] >= 25).astype(int)

In [17]:
print(df.groupby("PLAYER_NAME").size())

print(
    df[
        [
            "PLAYER_NAME",
            "GAME_DATE",
            "PTS",
            "PTS_L5",
            "PTS_L10",
            "PTS_STD_L5",
            "PTS_STD_10",
            "PTS_25_RATE_L10",
            "TARGET_25"
        ]
    ].tail(15)
)

PLAYER_NAME
Anthony Edwards            61
Devin Booker               64
Donovan Mitchell           70
Giannis Antetokounmpo      36
Jalen Brunson              74
Jayson Tatum               16
Luka Doncic                64
Nikola Jokic               65
Shai Gilgeous-Alexander    68
Stephen Curry              43
dtype: int64
         PLAYER_NAME  GAME_DATE  PTS  PTS_L5  PTS_L10  PTS_STD_L5  PTS_STD_10  \
546  Anthony Edwards 2026-02-20   40    26.4     28.2    8.142481    7.509993   
547  Anthony Edwards 2026-02-22   28    28.4     29.0   10.212737    8.339997   
548  Anthony Edwards 2026-02-24   34    27.0     29.8    9.539392    7.743097   
549  Anthony Edwards 2026-02-26   31    29.2     30.6    9.654015    7.720104   
550  Anthony Edwards 2026-03-01   21    29.4     30.4    9.685040    7.676805   
551  Anthony Edwards 2026-03-03   41    30.8     28.6    7.049823    7.545418   
552  Anthony Edwards 2026-03-05   22    31.0     29.7    7.382412    8.512083   
553  Anthony Edwards 2026-0

Create dynamic scoring threshold.

In [18]:
df["PROP_LINE"] = np.floor(df["PTS_L10"]) + 0.5

df["TARGET_OVER"] = np.where(
    df["PROP_LINE"].notna(),
    (df["PTS"] > df["PROP_LINE"]).astype(int),
    np.nan
)

Download data

In [19]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "multi_player.csv", index=False)